# NB02 — OLS Derivation from First Principles

> **Where does β₁ = Σ(x−x̄)(y−ȳ) / Σ(x−x̄)² come from?**

We derive it step by step using calculus — no black boxes.


## 1. Set up the objective

We want to find β₀ and β₁ that minimise:

```
SSR(β₀, β₁) = Σᵢ (yᵢ − β₀ − β₁xᵢ)²
```

This is a smooth bowl-shaped function.
At the minimum, both partial derivatives are **zero**.


## Step 1 — Partial derivative with respect to β₀

```
∂SSR/∂β₀ = -2 Σ (yᵢ − β₀ − β₁xᵢ) = 0

⟹  Σyᵢ = n·β₀ + β₁·Σxᵢ

⟹  β₀ = ȳ − β₁·x̄          ← Intercept formula
```

The fitted line **always passes through the point (x̄, ȳ)**.


## Step 2 — Partial derivative with respect to β₁

```
∂SSR/∂β₁ = -2 Σ xᵢ(yᵢ − β₀ − β₁xᵢ) = 0

⟹  Σ xᵢyᵢ = β₀·Σxᵢ + β₁·Σxᵢ²
```

Substitute β₀ = ȳ − β₁·x̄:

```
Σ xᵢyᵢ − n·x̄·ȳ = β₁ (Σxᵢ² − n·x̄²)

⟹  β₁ = Σ(xᵢ − x̄)(yᵢ − ȳ)  /  Σ(xᵢ − x̄)²    ← Slope formula
```


In [ ]:
import numpy as np

# Verify derivation on a small dataset
X = np.array([1., 2., 3., 4., 5.])
y = np.array([2., 4., 5., 4., 5.])

n    = len(X)
xbar = X.mean()
ybar = y.mean()

# Derived formula
b1 = np.sum((X - xbar) * (y - ybar)) / np.sum((X - xbar) ** 2)
b0 = ybar - b1 * xbar

print(f"x̄ = {xbar:.2f},  ȳ = {ybar:.2f}")
print(f"β₁ (slope)     = {b1:.6f}")
print(f"β₀ (intercept) = {b0:.6f}")
print(f"Line: ŷ = {b0:.4f} + {b1:.4f}·x")

# Cross-check with numpy polyfit
b1_np, b0_np = np.polyfit(X, y, 1)
print(f"\nnumpy polyfit: β₁={b1_np:.6f}, β₀={b0_np:.6f}")
print(f"Match: {np.isclose(b1, b1_np) and np.isclose(b0, b0_np)}")


## 3. Matrix form — generalises to multiple regression

For multiple predictors, the same idea becomes:

```
β̂ = (XᵀX)⁻¹ Xᵀy
```

Where **X** is the **design matrix** — one row per observation, columns are [1, x₁, x₂, ...].
The column of 1s captures the intercept β₀.


In [ ]:
import numpy as np

X_raw = np.array([1., 2., 3., 4., 5.])
y     = np.array([2., 4., 5., 4., 5.])

# Design matrix: column of 1s + X
X_design = np.column_stack([np.ones(len(X_raw)), X_raw])
print("Design matrix X:")
print(X_design)

# OLS normal equations: β̂ = (XᵀX)⁻¹ Xᵀy
XtX     = X_design.T @ X_design
Xty     = X_design.T @ y
beta    = np.linalg.inv(XtX) @ Xty

print(f"\nβ̂ = (XᵀX)⁻¹Xᵀy = {beta}")
print(f"β₀ = {beta[0]:.6f}  (intercept)")
print(f"β₁ = {beta[1]:.6f}  (slope)")


## 4. Geometric interpretation

The OLS solution projects **y** onto the **column space of X**:

```
ŷ = X · β̂  =  X(XᵀX)⁻¹Xᵀy  =  H · y
```

**H** is the **hat matrix** (it puts a "hat" on y).
ŷ is the closest point in the column space of X to the observed y —
closest measured by Euclidean distance (= minimising SSR).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X_raw = np.array([1., 2., 3., 4., 5.])
y     = np.array([2., 4., 5., 4., 5.])
X_design = np.column_stack([np.ones(len(X_raw)), X_raw])

# Hat matrix
H     = X_design @ np.linalg.inv(X_design.T @ X_design) @ X_design.T
y_hat = H @ y

residuals = y - y_hat
print("Fitted values ŷ:", np.round(y_hat, 3))
print("Residuals e    :", np.round(residuals, 3))
print(f"SSR            : {np.sum(residuals**2):.6f}")

# Verify: residuals are orthogonal to X columns
print(f"\nOrthogonality check (should be ~0):")
print(f"  Xᵀe = {X_design.T @ residuals}")  # should be [0, 0]

plt.figure(figsize=(7, 4))
plt.scatter(X_raw, y, s=80, color='steelblue', zorder=3, label='Observed y')
plt.plot(X_raw, y_hat, 'r-', linewidth=2, label='Fitted ŷ')
for xi, yi, yp in zip(X_raw, y, y_hat):
    plt.vlines(xi, min(yi,yp), max(yi,yp), color='gray', linewidth=1.5, linestyle='--')
plt.xlabel('x'); plt.ylabel('y')
plt.title('OLS fit via normal equations')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


## Key Takeaways

| Result | Formula |
|--------|---------|
| Slope | β₁ = Σ(xᵢ−x̄)(yᵢ−ȳ) / Σ(xᵢ−x̄)² |
| Intercept | β₀ = ȳ − β₁x̄ |
| Matrix form | β̂ = (XᵀX)⁻¹Xᵀy |
| Hat matrix | H = X(XᵀX)⁻¹Xᵀ |
| Orthogonality | Xᵀe = 0 always |

**Next:** NB03 — implement OLS completely from scratch in Python (no numpy linalg).
